In [1]:
import sys
from importlib.util import find_spec

print("Python do notebook:", sys.executable)

for modulo in ["spacy", "pt_core_news_sm"]:
    disponivel = find_spec(modulo) is not None
    print(f"{modulo}: {'disponível' if disponivel else 'não encontrado'}")

Python do notebook: /opt/homebrew/anaconda3/bin/python
spacy: disponível
pt_core_news_sm: disponível


In [3]:
import spacy

nlp = spacy.load("pt_core_news_sm")

print("Modelo carregado:", nlp.meta["name"])
print("Versão do spaCy:", spacy.__version__)
print("Versão do modelo:", nlp.meta["version"])
print("Categorias do NER:", nlp.get_pipe("ner").labels)

Modelo carregado: core_news_sm
Versão do spaCy: 3.8.16
Versão do modelo: 3.8.0
Categorias do NER: ('LOC', 'MISC', 'ORG', 'PER')


In [4]:
texto = "Como solicitar o trancamento de matrícula no Ibmec Faria Lima?"

doc = nlp(texto)

print("Texto:", texto)
print("\nEntidades encontradas:")

if doc.ents:
    for entidade in doc.ents:
        print(
            entidade.text,
            "->", entidade.label_,
            "| posição:", entidade.start_char, entidade.end_char,
        )
else:
    print("Nenhuma entidade encontrada.")

Texto: Como solicitar o trancamento de matrícula no Ibmec Faria Lima?

Entidades encontradas:
Ibmec Faria Lima -> LOC | posição: 45 61


In [5]:
from spacy import displacy

displacy.render(doc, style="ent", jupyter=True)

In [6]:
# Pipeline em português, inicialmente sem modelo treinado.
nlp_regras = spacy.blank("pt")

ruler = nlp_regras.add_pipe(
    "entity_ruler",
    config={"phrase_matcher_attr": "LOWER"},
)

ruler.add_patterns([
    {
        "label": "SERVICO_ACADEMICO",
        "pattern": "matrícula",
    }
])

doc_regras = nlp_regras(texto)

for entidade in doc_regras.ents:
    print(entidade.text, "->", entidade.label_)

matrícula -> SERVICO_ACADEMICO


In [7]:
ruler.add_patterns([
    {"label": "INSTITUICAO", "pattern": "Ibmec"},
    {"label": "UNIDADE", "pattern": "Faria Lima"},
])

for nome, pipeline in [
    ("MODELO PRÉ-TREINADO", nlp),
    ("REGRAS DO PROJETO", nlp_regras),
]:
    print(f"\n{nome}")

    documento = pipeline(texto)

    for entidade in documento.ents:
        print(entidade.text, "->", entidade.label_)


MODELO PRÉ-TREINADO
Ibmec Faria Lima -> LOC

REGRAS DO PROJETO
matrícula -> SERVICO_ACADEMICO
Ibmec -> INSTITUICAO
Faria Lima -> UNIDADE


In [8]:
perguntas_exploratorias = [
    "Como cancelar <minha matrícula?",
    "Como trancar minha MATRÍCULA no IBMEC FARIA LIMA?",
    "Como reabrir minha matricula?",
    "Onde consulto as matrículas anteriores?",
    "Como conseguir um estágio no Ibmec?",
    "O prazo de matrícula já terminou?",
]

for pergunta in perguntas_exploratorias:
    documento = nlp_regras(pergunta)

    entidades = [
        (entidade.text, entidade.label_)
        for entidade in documento.ents
    ]

    print("\nPergunta:", pergunta)
    print("Entidades:", entidades)


Pergunta: Como cancelar <minha matrícula?
Entidades: [('matrícula', 'SERVICO_ACADEMICO')]

Pergunta: Como trancar minha MATRÍCULA no IBMEC FARIA LIMA?
Entidades: [('MATRÍCULA', 'SERVICO_ACADEMICO'), ('IBMEC', 'INSTITUICAO'), ('FARIA LIMA', 'UNIDADE')]

Pergunta: Como reabrir minha matricula?
Entidades: []

Pergunta: Onde consulto as matrículas anteriores?
Entidades: []

Pergunta: Como conseguir um estágio no Ibmec?
Entidades: [('Ibmec', 'INSTITUICAO')]

Pergunta: O prazo de matrícula já terminou?
Entidades: [('matrícula', 'SERVICO_ACADEMICO')]


In [9]:
variantes_matricula = [
    "matricula",
    "matrículas",
    "matriculas",
]

ruler.add_patterns([
    {
        "label": "SERVICO_ACADEMICO",
        "pattern": variante,
    }
    for variante in variantes_matricula
])

for pergunta in perguntas_exploratorias:
    documento = nlp_regras(pergunta)

    print("\nPergunta:", pergunta)
    print("Entidades:", [
        (entidade.text, entidade.label_)
        for entidade in documento.ents
    ])


Pergunta: Como cancelar <minha matrícula?
Entidades: [('matrícula', 'SERVICO_ACADEMICO')]

Pergunta: Como trancar minha MATRÍCULA no IBMEC FARIA LIMA?
Entidades: [('MATRÍCULA', 'SERVICO_ACADEMICO'), ('IBMEC', 'INSTITUICAO'), ('FARIA LIMA', 'UNIDADE')]

Pergunta: Como reabrir minha matricula?
Entidades: [('matricula', 'SERVICO_ACADEMICO')]

Pergunta: Onde consulto as matrículas anteriores?
Entidades: [('matrículas', 'SERVICO_ACADEMICO')]

Pergunta: Como conseguir um estágio no Ibmec?
Entidades: [('Ibmec', 'INSTITUICAO')]

Pergunta: O prazo de matrícula já terminou?
Entidades: [('matrícula', 'SERVICO_ACADEMICO')]


In [10]:
termos_academicos = [
    "estágio",
    "estagio",
    "estágios",
    "estagios",
    "carteirinha",
    "carteirinhas",
]

ruler.add_patterns([
    {"label": "SERVICO_ACADEMICO", "pattern": termo}
    for termo in termos_academicos
])

novas_perguntas = [
    "Como conseguir um estágio no Ibmec?",
    "Onde envio os documentos de estagio?",
    "Como solicitar minha carteirinha na Faria Lima?",
    "Posso usar carteirinhas digitais?",
]

for pergunta in novas_perguntas:
    documento = nlp_regras(pergunta)

    print("\nPergunta:", pergunta)
    print("Entidades:", [
        (entidade.text, entidade.label_)
        for entidade in documento.ents
    ])


Pergunta: Como conseguir um estágio no Ibmec?
Entidades: [('estágio', 'SERVICO_ACADEMICO'), ('Ibmec', 'INSTITUICAO')]

Pergunta: Onde envio os documentos de estagio?
Entidades: [('estagio', 'SERVICO_ACADEMICO')]

Pergunta: Como solicitar minha carteirinha na Faria Lima?
Entidades: [('carteirinha', 'SERVICO_ACADEMICO'), ('Faria Lima', 'UNIDADE')]

Pergunta: Posso usar carteirinhas digitais?
Entidades: [('carteirinhas', 'SERVICO_ACADEMICO')]


In [11]:
SERVICO = "SERVICO_ACADEMICO"

casos_avaliacao = [
    (
        "Preciso reabrir a matrícula no Ibmec.",
        [("matrícula", SERVICO), ("Ibmec", "INSTITUICAO")],
    ),
    (
        "Na Faria Lima, como peço minha carteirinha?",
        [("Faria Lima", "UNIDADE"), ("carteirinha", SERVICO)],
    ),
    (
        "Quais documentos são exigidos para estágios?",
        [("estágios", SERVICO)],
    ),
    (
        "Há orientações sobre matriculas no IBMEC?",
        [("matriculas", SERVICO), ("IBMEC", "INSTITUICAO")],
    ),
    (
        "Preciso renovar a matrícula e a carteirinha.",
        [("matrícula", SERVICO), ("carteirinha", SERVICO)],
    ),
    ("O sistema matricula os alunos automaticamente.", []),
    ("Eu estagio em uma empresa parceira.", []),
    ("Como posso obter ajuda?", []),
]

for numero, (frase, esperado) in enumerate(casos_avaliacao, start=1):
    print(f"\n{numero}. {frase}")
    print("Esperado:", esperado)


1. Preciso reabrir a matrícula no Ibmec.
Esperado: [('matrícula', 'SERVICO_ACADEMICO'), ('Ibmec', 'INSTITUICAO')]

2. Na Faria Lima, como peço minha carteirinha?
Esperado: [('Faria Lima', 'UNIDADE'), ('carteirinha', 'SERVICO_ACADEMICO')]

3. Quais documentos são exigidos para estágios?
Esperado: [('estágios', 'SERVICO_ACADEMICO')]

4. Há orientações sobre matriculas no IBMEC?
Esperado: [('matriculas', 'SERVICO_ACADEMICO'), ('IBMEC', 'INSTITUICAO')]

5. Preciso renovar a matrícula e a carteirinha.
Esperado: [('matrícula', 'SERVICO_ACADEMICO'), ('carteirinha', 'SERVICO_ACADEMICO')]

6. O sistema matricula os alunos automaticamente.
Esperado: []

7. Eu estagio em uma empresa parceira.
Esperado: []

8. Como posso obter ajuda?
Esperado: []


In [12]:
from collections import Counter

totais_ner = Counter()
resultados_avaliacao = []

for numero, (frase, anotacoes) in enumerate(casos_avaliacao, start=1):
    esperados = set()

    for trecho, rotulo in anotacoes:
        # Estes exemplos têm uma única ocorrência de cada trecho.
        assert frase.count(trecho) == 1

        inicio = frase.index(trecho)
        esperados.add((inicio, inicio + len(trecho), rotulo))

    documento = nlp_regras(frase)

    previstos = {
        (entidade.start_char, entidade.end_char, entidade.label_)
        for entidade in documento.ents
    }

    acertos = previstos & esperados
    falsos_positivos = previstos - esperados
    falsos_negativos = esperados - previstos

    contagens = {
        "VP": len(acertos),
        "FP": len(falsos_positivos),
        "FN": len(falsos_negativos),
    }
    totais_ner.update(contagens)

    resultados_avaliacao.append({
        "texto": frase,
        "esperados": esperados,
        "previstos": previstos,
        **contagens,
    })

    print(f"\n{numero}. {frase}")
    print(contagens)

    for inicio, fim, rotulo in sorted(falsos_positivos):
        print("Reconheceu indevidamente:", frase[inicio:fim], "->", rotulo)

    for inicio, fim, rotulo in sorted(falsos_negativos):
        print("Deixou de reconhecer:", frase[inicio:fim], "->", rotulo)

print("\nTOTAIS:", dict(totais_ner))


1. Preciso reabrir a matrícula no Ibmec.
{'VP': 2, 'FP': 0, 'FN': 0}

2. Na Faria Lima, como peço minha carteirinha?
{'VP': 2, 'FP': 0, 'FN': 0}

3. Quais documentos são exigidos para estágios?
{'VP': 1, 'FP': 0, 'FN': 0}

4. Há orientações sobre matriculas no IBMEC?
{'VP': 2, 'FP': 0, 'FN': 0}

5. Preciso renovar a matrícula e a carteirinha.
{'VP': 2, 'FP': 0, 'FN': 0}

6. O sistema matricula os alunos automaticamente.
{'VP': 0, 'FP': 1, 'FN': 0}
Reconheceu indevidamente: matricula -> SERVICO_ACADEMICO

7. Eu estagio em uma empresa parceira.
{'VP': 0, 'FP': 1, 'FN': 0}
Reconheceu indevidamente: estagio -> SERVICO_ACADEMICO

8. Como posso obter ajuda?
{'VP': 0, 'FP': 0, 'FN': 0}

TOTAIS: {'VP': 9, 'FP': 2, 'FN': 0}


In [13]:
vp = totais_ner["VP"]
fp = totais_ner["FP"]
fn = totais_ner["FN"]

precisao = vp / (vp + fp) if vp + fp else 0.0
revocacao = vp / (vp + fn) if vp + fn else 0.0

f1 = (
    2 * precisao * revocacao / (precisao + revocacao)
    if precisao + revocacao else 0.0
)

print(f"Precisão: {precisao:.2%}")
print(f"Revocação: {revocacao:.2%}")
print(f"F1: {f1:.2%}")

Precisão: 81.82%
Revocação: 100.00%
F1: 90.00%


In [14]:
rotulos = ["INSTITUICAO", "UNIDADE", "SERVICO_ACADEMICO"]
metricas_por_rotulo = []

for rotulo in rotulos:
    vp = fp = fn = 0

    for resultado in resultados_avaliacao:
        esperados = {
            entidade for entidade in resultado["esperados"]
            if entidade[2] == rotulo
        }
        previstos = {
            entidade for entidade in resultado["previstos"]
            if entidade[2] == rotulo
        }

        vp += len(previstos & esperados)
        fp += len(previstos - esperados)
        fn += len(esperados - previstos)

    precisao_rotulo = vp / (vp + fp) if vp + fp else 0.0
    revocacao_rotulo = vp / (vp + fn) if vp + fn else 0.0
    f1_rotulo = 2 * vp / (2 * vp + fp + fn) if 2 * vp + fp + fn else 0.0

    metricas_por_rotulo.append({
        "rotulo": rotulo,
        "VP": vp,
        "FP": fp,
        "FN": fn,
        "suporte": vp + fn,
        "precisao": precisao_rotulo,
        "revocacao": revocacao_rotulo,
        "f1": f1_rotulo,
    })

    print(
        f"{rotulo} | suporte={vp + fn} | VP={vp} FP={fp} FN={fn} | "
        f"P={precisao_rotulo:.2%} R={revocacao_rotulo:.2%} "
        f"F1={f1_rotulo:.2%}"
    )

INSTITUICAO | suporte=2 | VP=2 FP=0 FN=0 | P=100.00% R=100.00% F1=100.00%
UNIDADE | suporte=1 | VP=1 FP=0 FN=0 | P=100.00% R=100.00% F1=100.00%
SERVICO_ACADEMICO | suporte=6 | VP=6 FP=2 FN=0 | P=75.00% R=100.00% F1=85.71%


In [15]:
padroes_congelados = [
    {"label": rotulo, "pattern": padrao}
    for rotulo, padrao in sorted({
        (regra["label"], regra["pattern"])
        for regra in ruler.patterns
    })
]

configuracao_experimento = {
    "versao_experimento": "ner_basico_v1",
    "abordagem_avaliada": "Regras com EntityRuler",
    "idioma": "pt",
    "comparacao_texto": "LOWER",
    "versao_spacy": spacy.__version__,
    "modelo_pre_treinado_comparado": nlp.meta["name"],
    "versao_modelo_pre_treinado": nlp.meta["version"],
    "quantidade_casos": len(casos_avaliacao),
    "tipo_avaliacao": "Piloto com exemplos sintéticos",
    "origem_anotacoes": "Propostas com apoio de IA",
}

print("Regras distintas:", len(padroes_congelados))
print("Casos avaliados:", configuracao_experimento["quantidade_casos"])
print("Abordagem avaliada:", configuracao_experimento["abordagem_avaliada"])

Regras distintas: 12
Casos avaliados: 8
Abordagem avaliada: Regras com EntityRuler


In [16]:
from pathlib import Path
import csv

pasta_atual = Path.cwd()
raiz = pasta_atual.parent if pasta_atual.name == "notebooks" else pasta_atual
pasta_resultados = raiz / "reports" / "ner_basico_v1"

linhas_casos = []
linhas_entidades = []

for numero, resultado in enumerate(resultados_avaliacao, start=1):
    linhas_casos.append({
        "caso": numero,
        "texto": resultado["texto"],
        "VP": resultado["VP"],
        "FP": resultado["FP"],
        "FN": resultado["FN"],
    })

    for origem, campo in [
        ("referencia", "esperados"),
        ("previsao", "previstos"),
    ]:
        for inicio, fim, rotulo in sorted(resultado[campo]):
            linhas_entidades.append({
                "caso": numero,
                "origem": origem,
                "inicio": inicio,
                "fim": fim,
                "texto_entidade": resultado["texto"][inicio:fim],
                "rotulo": rotulo,
            })

vp, fp, fn = (
    totais_ner[chave] for chave in ["VP", "FP", "FN"]
)

metrica_global = {
    "rotulo": "GLOBAL_MICRO",
    "VP": vp,
    "FP": fp,
    "FN": fn,
    "suporte": vp + fn,
    "precisao": vp / (vp + fp) if vp + fp else 0.0,
    "revocacao": vp / (vp + fn) if vp + fn else 0.0,
    "f1": 2 * vp / (2 * vp + fp + fn) if 2 * vp + fp + fn else 0.0,
}

configuracao_salva = configuracao_experimento.copy()
configuracao_salva["revisao_humana_anotacoes"] = "Não registrada"
configuracao_salva["quantidade_padroes"] = len(padroes_congelados)

saidas_ner = {
    "padroes.csv": padroes_congelados,
    "casos.csv": linhas_casos,
    "entidades.csv": linhas_entidades,
    "metricas.csv": [metrica_global] + metricas_por_rotulo,
    "configuracao.csv": [configuracao_salva],
}

pasta_resultados.mkdir(parents=True, exist_ok=True)

for nome in saidas_ner:
    if (pasta_resultados / nome).exists():
        raise FileExistsError(f"{nome} já existe; nada foi sobrescrito.")

for nome, linhas in saidas_ner.items():
    colunas = list(dict.fromkeys(
        campo for linha in linhas for campo in linha
    ))
    esperadas = [
        {campo: str(linha.get(campo, "")) for campo in colunas}
        for linha in linhas
    ]

    caminho = pasta_resultados / nome

    with caminho.open("x", encoding="utf-8-sig", newline="") as arquivo:
        escritor = csv.DictWriter(
            arquivo, fieldnames=colunas, delimiter=";"
        )
        escritor.writeheader()
        escritor.writerows(esperadas)

    with caminho.open("r", encoding="utf-8-sig", newline="") as arquivo:
        recuperadas = list(csv.DictReader(arquivo, delimiter=";"))

    assert recuperadas == esperadas, f"Falha na gravação de {nome}"
    print(f"{nome}: {len(recuperadas)} registros — conferido")

padroes.csv: 12 registros — conferido
casos.csv: 8 registros — conferido
entidades.csv: 20 registros — conferido
metricas.csv: 4 registros — conferido
configuracao.csv: 1 registros — conferido


In [17]:
def ler_resultado_ner(nome):
    with (pasta_resultados / nome).open(
        "r", encoding="utf-8-sig", newline=""
    ) as arquivo:
        return list(csv.DictReader(arquivo, delimiter=";"))


configuracao_lida = ler_resultado_ner("configuracao.csv")[0]
padroes_lidos = ler_resultado_ner("padroes.csv")
casos_lidos = ler_resultado_ner("casos.csv")
entidades_lidas = ler_resultado_ner("entidades.csv")

nlp_recarregado = spacy.blank(configuracao_lida["idioma"])

ruler_recarregado = nlp_recarregado.add_pipe(
    "entity_ruler",
    config={
        "phrase_matcher_attr": configuracao_lida["comparacao_texto"]
    },
)
ruler_recarregado.add_patterns(padroes_lidos)

for caso in casos_lidos:
    previsoes_salvas = {
        (int(entidade["inicio"]), int(entidade["fim"]), entidade["rotulo"])
        for entidade in entidades_lidas
        if entidade["caso"] == caso["caso"]
        and entidade["origem"] == "previsao"
    }

    documento = nlp_recarregado(caso["texto"])

    previsoes_reproduzidas = {
        (entidade.start_char, entidade.end_char, entidade.label_)
        for entidade in documento.ents
    }

    assert previsoes_reproduzidas == previsoes_salvas, (
        f"Previsões diferentes no caso {caso['caso']}"
    )

print("Reprodução conferida nos 8 casos.")

Reprodução conferida nos 8 casos.
